In [ ]:
%pip install python-dotenv

In [ ]:
from pathlib import Path
project_root = Path.cwd()

import os
import requests
from urllib.parse import unquote
from dotenv import load_dotenv

load_dotenv(project_root / ".env", override=True)

# .env에서 Encoding 인증키 불러오기
encoded_key = os.getenv("DATA_GO_KR_API_KEY", "").strip()

# URL Encoding 해제
API_KEY = unquote(encoded_key)

BASE_URL = (
    "http://apis.data.go.kr/6260000/"
    "BusanCommercialChangeService/"
    "getCommercialChangeList"
)

params = {
    "serviceKey": API_KEY,
    "numOfRows": 5,
    "pageNo": 1
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=30
)

print("HTTP Status:", response.status_code)
print("Response:")
print(response.text[:2000])

In [ ]:
import math
import time
import pandas as pd
import xml.etree.ElementTree as ET

# API 응답 파싱
def fetch_page(page_no, num_rows=100):
    params = {
        "serviceKey": API_KEY,
        "numOfRows": num_rows,
        "pageNo": page_no
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=30
    )
    response.raise_for_status()

    root = ET.fromstring(response.content)

    result_code = root.findtext(".//resultCode")
    if result_code != "00":
        raise RuntimeError(
            f"API 오류: {result_code}, "
            f"{root.findtext('.//resultMsg')}"
        )

    total_count = int(root.findtext(".//totalCount", "0"))

    records = []
    for item in root.findall(".//items/item"):
        record = {
            child.tag: child.text
            for child in item
        }
        records.append(record)

    return records, total_count


# 첫 페이지 수집
page_size = 100
first_records, total_count = fetch_page(1, page_size)

all_records = first_records.copy()
total_pages = math.ceil(total_count / page_size)

print(f"전체 데이터: {total_count}건")
print(f"전체 페이지: {total_pages}페이지")

# 나머지 페이지 수집
for page in range(2, total_pages + 1):
    records, _ = fetch_page(page, page_size)
    all_records.extend(records)

    print(f"{page}/{total_pages}페이지 수집 완료")
    time.sleep(0.2)

# DataFrame 생성
df = pd.DataFrame(all_records)

print(f"\n수집 완료: {len(df)}건")
display(df.head())